In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR1TOT_E.xpt"
csv_file = DATA_DIR / "DR1TOT_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 DR1TOT_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = """
SEQN
WTDRD1
WTDR2D
DR1DRSTZ
DR1EXMER
DRABF
DRDINT
DR1DBIH
DR1DAY
DR1LANG
DR1MNRSP
DR1HELPD
DBQ095Z
DBD100
DRQSPREP
DRQSDIET
DRQSDT1
DRQSDT2
DRQSDT3
DRQSDT4
DRQSDT5
DRQSDT6
DRQSDT7
DRQSDT8
DRQSDT9
DRQSDT10
DRQSDT91
DR1TNUMF
DR1TKCAL
DR1TPROT
DR1TCARB
DR1TSUGR
DR1TFIBE
DR1TTFAT
DR1TSFAT
DR1TMFAT
DR1TPFAT
DR1TCHOL
DR1TATOC
DR1TATOA
DR1TRET
DR1TVARA
DR1TACAR
DR1TBCAR
DR1TCRYP
DR1TLYCO
DR1TLZ
DR1TVB1
DR1TVB2
DR1TNIAC
DR1TVB6
DR1TFOLA
DR1TFA
DR1TFF
DR1TFDFE
DR1TCHL
DR1TVB12
DR1TB12A
DR1TVC
DR1TVD
DR1TVK
DR1TCALC
DR1TPHOS
DR1TMAGN
DR1TIRON
DR1TZINC
DR1TCOPP
DR1TSODI
DR1TPOTA
DR1TSELE
DR1TCAFF
DR1TTHEO
DR1TALCO
DR1TMOIS
DR1TS040
DR1TS060
DR1TS080
DR1TS100
DR1TS120
DR1TS140
DR1TS160
DR1TS180
DR1TM161
DR1TM181
DR1TM201
DR1TM221
DR1TP182
DR1TP183
DR1TP184
DR1TP204
DR1TP205
DR1TP225
DR1TP226
DR1_300
DR1_320Z
DR1_330Z
DR1BWATZ
DR1TWS
DRD340
DRD350A
DRD350AQ
DRD350B
DRD350BQ
DRD350C
DRD350CQ
DRD350D
DRD350DQ
DRD350E
DRD350EQ
DRD350F
DRD350FQ
DRD350G
DRD350GQ
DRD350H
DRD350HQ
DRD350I
DRD350IQ
DRD350J
DRD350JQ
DRD350K
DRD360
DRD370A
DRD370AQ
DRD370B
DRD370BQ
DRD370C
DRD370CQ
DRD370D
DRD370DQ
DRD370E
DRD370EQ
DRD370F
DRD370FQ
DRD370G
DRD370GQ
DRD370H
DRD370HQ
DRD370I
DRD370IQ
DRD370J
DRD370JQ
DRD370K
DRD370KQ
DRD370L
DRD370LQ
DRD370M
DRD370MQ
DRD370N
DRD370NQ
DRD370O
DRD370OQ
DRD370P
DRD370PQ
DRD370Q
DRD370QQ
DRD370R
DRD370RQ
DRD370S
DRD370SQ
DRD370T
DRD370TQ
DRD370U
DRD370UQ
DRD370V
""".split()


if df.shape != (9762, 164):

    raise ValueError(
        f"Unexpected DR1TOT_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DR1TOT_E columns/order differ."
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: DR1TOT_E structure = "
    "9,762 rows x 164 columns."
)


# ============================================================
# 4. DEFINE DIETARY MEASUREMENT FIELDS
# ============================================================

#
# Preserve these EXACTLY as XPT-decoded floats.
#
# This includes:
#
# - dietary sample weights
# - all nutrient totals
# - water quantities
#
# DO NOT:
#
# - Int64 them
# - round them
# - trim .0
# - use float_format
#

NUTRIENT_COLUMNS = (
    df.columns[
        df.columns.get_loc("DR1TKCAL"):
        df.columns.get_loc("DR1TP226") + 1
    ]
    .tolist()
)


MEASUREMENT_COLUMNS = (
    [
        "WTDRD1",
        "WTDR2D"
    ]
    +
    NUTRIENT_COLUMNS
    +
    [
        "DR1_320Z",
        "DR1_330Z",
        "DR1BWATZ"
    ]
)


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in MEASUREMENT_COLUMNS
]


print(
    "\nDietary measurement fields preserved AS IS:",
    len(MEASUREMENT_COLUMNS)
)

print(
    "Nutrient fields:",
    len(NUTRIENT_COLUMNS)
)

print(
    "Integer/code/count fields:",
    len(INTEGER_COLUMNS)
)


if len(MEASUREMENT_COLUMNS) != 70:

    raise ValueError(
        "Unexpected measurement-field count."
    )


if len(NUTRIENT_COLUMNS) != 65:

    raise ValueError(
        "Unexpected nutrient-field count."
    )


# ============================================================
# 5. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifacts:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. EXPECTED TINY COUNTS FROM ATTACHED XPT
# ============================================================

EXPECTED_TINY_COUNTS = {

    "DR1TATOA": 8184,
    "DR1TALCO": 7666,
    "DR1BWATZ": 6034,
    "DR1TB12A": 5916,
    "DR1TP184": 5776,
    "DR1_330Z": 5264,
    "DR1TTHEO": 4472,
    "DR1TP225": 3576,
    "DR1TM221": 2879,
    "DR1_320Z": 2662,
    "DR1TP205": 2461,
    "DR1TLYCO": 2433,
    "DR1TCAFF": 2179,
    "DR1TP226": 2005,
    "DR1TACAR": 1161,
    "DR1TCRYP": 994,
    "DR1TS040": 554,
    "DR1TS060": 503,

    # Day difference field:
    "DR1DBIH": 432,

    "DR1TP204": 346,
    "DR1TS080": 293,
    "DR1TFA": 144,
    "DR1TBCAR": 128,
    "DR1TLZ": 118,
    "DR1TVD": 112,
    "DR1TM201": 102,
    "DR1TS100": 100,
    "DR1TFIBE": 98,
    "DR1TFF": 82,
    "DR1TRET": 79,
    "DR1TCHOL": 53,
    "DR1TVC": 42,
    "DR1TM161": 34,
    "DR1TS120": 30,
    "DR1TVARA": 22,
    "DR1TVB12": 22,
    "DR1TS140": 12,
    "DR1TVK": 10,
    "DR1TATOC": 9,
    "DR1TM181": 9,
    "DR1TS180": 9,
    "DR1TP183": 7,
    "DR1TP182": 7,
    "DR1TPFAT": 6,
    "DR1TVB6": 6,
    "DR1TTFAT": 6,
    "DR1TS160": 6,
    "DR1TVB1": 6,
    "DR1TSFAT": 6,
    "DR1TMFAT": 6,
    "DR1TFDFE": 5,
    "DR1TFOLA": 5,
    "DR1TVB2": 4,
    "DR1TSELE": 4,
    "DR1TSUGR": 3,
    "DR1TPHOS": 3,
    "DR1TCOPP": 2,

    # Number of foods:
    # CDC 2007-2008 range is 0-52.
    "DR1TNUMF": 1,

    "DR1TCARB": 1,
    "DR1TPROT": 1,
    "DR1TKCAL": 1,
    "DR1TCHL": 1,
    "DR1TMAGN": 1,
    "DR1TCALC": 1,
    "DR1TNIAC": 1,
    "DR1TMOIS": 1,
    "DR1TSODI": 1,
    "DR1TZINC": 1,
    "DR1TIRON": 1,
    "DR1TPOTA": 1
}


EXPECTED_TINY_TOTAL = 67100


# ============================================================
# 7. VALIDATE TINY ARTIFACT LOCATIONS
# ============================================================

actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    missing = (
        set(EXPECTED_TINY_COUNTS)
        -
        set(actual_tiny_counts)
    )

    unexpected = (
        set(actual_tiny_counts)
        -
        set(EXPECTED_TINY_COUNTS)
    )

    different_counts = {
        col: {
            "expected": EXPECTED_TINY_COUNTS.get(col),
            "actual": actual_tiny_counts.get(col)
        }
        for col in (
            set(EXPECTED_TINY_COUNTS)
            &
            set(actual_tiny_counts)
        )
        if (
            EXPECTED_TINY_COUNTS[col]
            !=
            actual_tiny_counts[col]
        )
    }


    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Missing variables: {sorted(missing)}\n"
        f"Unexpected variables: {sorted(unexpected)}\n"
        f"Different counts: {different_counts}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny artifact count."
    )


if len(
    actual_tiny_counts
) != 70:

    raise ValueError(
        "Unexpected number of tiny-value variables."
    )


print(
    "\nPASS: Exactly 67,100 tiny artifacts "
    "across 70 validated variables."
)


# ============================================================
# 8. ZERO SAFETY CHECK
# ============================================================

#
# Tiny values are allowed ONLY in:
#
# - DR1DBIH
# - DR1TNUMF
# - nutrient measurement fields
# - water quantity fields
#
# If a tiny value appears anywhere else, STOP.
#

APPROVED_ZERO_COLUMNS = (
    set(
        NUTRIENT_COLUMNS
    )
    |
    {
        "DR1DBIH",
        "DR1TNUMF",
        "DR1_320Z",
        "DR1_330Z",
        "DR1BWATZ"
    }
)


unexpected_zero_artifact_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_zero_artifact_columns:

    raise ValueError(
        "STOP: Tiny values found in "
        "non-approved variables:\n"
        + str(
            sorted(
                unexpected_zero_artifact_columns
            )
        )
    )


print(
    "PASS: All tiny artifacts occur only "
    "in CDC zero-permitted variables."
)


# ============================================================
# 9. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: tiny count changed."
        )


    df.loc[
        mask,
        col
    ] = 0.0


print(
    "PASS: 67,100 validated tiny values "
    "restored to zero."
)


# ============================================================
# 10. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 11. SPECIAL ZERO CHECK - DR1DBIH
# ============================================================

#
# CDC:
#
# Number of days between intake and
# household interview
#
# Range = -60 to 73
# Valid = 9,273
# Missing = 489
#
# 0 is therefore valid.
#

s = df["DR1DBIH"]


if int(
    s.notna()
    .sum()
) != 9273:

    raise ValueError(
        "DR1DBIH valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 489:

    raise ValueError(
        "DR1DBIH missing count differs."
    )


if not (
    s.dropna()
    .between(
        -60,
        73
    )
    .all()
):

    raise ValueError(
        "DR1DBIH outside CDC range."
    )


if int(
    s.eq(0)
    .sum()
) != 432:

    raise ValueError(
        "DR1DBIH restored zero count differs."
    )


print(
    "\nPASS: DR1DBIH"
)

print(
    "      zeros = 432"
)

print(
    "      range = -60 to 73"
)


# ============================================================
# 12. SPECIAL ZERO CHECK - DR1TNUMF
# ============================================================

#
# IMPORTANT:
#
# Unlike the prior cycle,
# CDC 2007-2008 publishes:
#
# DR1TNUMF:
# Range 0-52
# Valid 9,255
# Missing 507
#
# Therefore the one tiny value IS a valid zero.
#

s = df["DR1TNUMF"]


if int(
    s.notna()
    .sum()
) != 9255:

    raise ValueError(
        "DR1TNUMF valid count differs."
    )


if int(
    s.isna()
    .sum()
) != 507:

    raise ValueError(
        "DR1TNUMF missing count differs."
    )


if not (
    s.dropna()
    .between(
        0,
        52
    )
    .all()
):

    raise ValueError(
        "DR1TNUMF outside CDC range."
    )


if int(
    s.eq(0)
    .sum()
) != 1:

    raise ValueError(
        "DR1TNUMF restored zero count differs."
    )


print(
    "PASS: DR1TNUMF"
)

print(
    "      zero = 1"
)

print(
    "      range = 0 to 52"
)


# ============================================================
# 13. SEQN VALIDATION
# ============================================================

print("\n--- SEQN CHECK ---")


seqn = df["SEQN"]


if int(
    seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN = 41475-51623, "
    "no missing, no duplicates."
)


# ============================================================
# 14. DIETARY WEIGHT VALIDATION
# ============================================================

#
# CDC:
#
# WTDRD1 valid = 9,255
# WTDR2D valid = 7,838
#

if int(
    df["WTDRD1"]
    .notna()
    .sum()
) != 9255:

    raise ValueError(
        "WTDRD1 valid count differs."
    )


if int(
    df["WTDR2D"]
    .notna()
    .sum()
) != 7838:

    raise ValueError(
        "WTDR2D valid count differs."
    )


if not (
    df["WTDRD1"]
    .dropna()
    .gt(0)
    .all()
):

    raise ValueError(
        "Unexpected WTDRD1 value."
    )


if not (
    df["WTDR2D"]
    .dropna()
    .gt(0)
    .all()
):

    raise ValueError(
        "Unexpected WTDR2D value."
    )


print(
    "\nPASS: Dietary weights validated."
)

print(
    "      WTDRD1 valid = 9,255"
)

print(
    "      WTDR2D valid = 7,838"
)


# ============================================================
# 15. DR1DRSTZ CDC FREQUENCY CHECK
# ============================================================

EXPECTED_DR1DRSTZ = {
    1: 9118,
    2: 83,
    4: 137,
    5: 424
}


actual = (
    df["DR1DRSTZ"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_DR1DRSTZ:

    raise ValueError(
        "DR1DRSTZ frequency mismatch.\n"
        f"Expected: {EXPECTED_DR1DRSTZ}\n"
        f"Actual:   {actual}"
    )


print(
    "PASS: DR1DRSTZ matches CDC."
)


# ============================================================
# 16. NUTRIENT VALID COUNT CHECK
# ============================================================

#
# All 65 nutrient totals in this file should
# have 9,118 valid records.
#

nutrient_valid_failures = {}


for col in NUTRIENT_COLUMNS:

    valid_count = int(
        df[col]
        .notna()
        .sum()
    )


    if valid_count != 9118:

        nutrient_valid_failures[
            col
        ] = valid_count


if nutrient_valid_failures:

    raise ValueError(
        "Unexpected nutrient valid counts:\n"
        + str(
            nutrient_valid_failures
        )
    )


print(
    "PASS: All 65 nutrient fields "
    "have 9,118 valid values."
)


# ============================================================
# 17. WATER VARIABLE VALID COUNT CHECK
# ============================================================

for col in [
    "DR1_320Z",
    "DR1_330Z",
    "DR1BWATZ"
]:

    valid_count = int(
        df[col]
        .notna()
        .sum()
    )


    if valid_count != 9255:

        raise ValueError(
            f"{col}: expected 9,255 valid values, "
            f"found {valid_count:,}."
        )


print(
    "PASS: Water quantity fields "
    "have 9,255 valid values."
)


# ============================================================
# 18. VERIFY FRACTIONS ARE ONLY IN MEASUREMENT FIELDS
# ============================================================

print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)


fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    if count > 0:

        fractional_columns[col] = count


unexpected_fractional_columns = (
    set(
        fractional_columns
    )
    -
    set(
        MEASUREMENT_COLUMNS
    )
)


if unexpected_fractional_columns:

    raise ValueError(
        "STOP: Fractional values found "
        "outside dietary measurement fields:\n"
        + str(
            sorted(
                unexpected_fractional_columns
            )
        )
    )


print(
    "Fields containing genuine fractional values:",
    len(fractional_columns)
)


print(
    "PASS: Every genuine fractional value "
    "belongs to a dietary measurement field."
)


# ============================================================
# 19. VERIFY INTEGER/CODE FIELDS ARE WHOLE NUMBERS
# ============================================================

integer_fraction_failures = {}


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if count > 0:

        integer_fraction_failures[
            col
        ] = count


if integer_fraction_failures:

    raise ValueError(
        "STOP: Whole-number-designated fields "
        "contain fractional values:\n"
        + str(
            integer_fraction_failures
        )
    )


print(
    "PASS: All identifier/code/count fields "
    "are true whole-number values."
)


# ============================================================
# 20. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY the 67,100 validated tiny -> zero
# restorations.
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. CLEAN ONLY INTEGER / CODE / COUNT FIELDS
# ============================================================

#
# SEQN:
# 41475.0 -> 41475
#
# DR1DRSTZ:
# 1.0 -> 1
#
# DR1TNUMF:
# 12.0 -> 12
#
# DR1DBIH:
# 0.0 -> 0
#
#
# DO NOT Int64:
#
# weights
# nutrients
# water quantities
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN/code/count fields "
    "converted to clean integer formatting."
)

print(
    "PASS: All 70 dietary measurement fields "
    "remain source-decoded floats."
)


# ============================================================
# 22. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

print(
    "\n--- PRE-EXPORT NUMERIC FIDELITY ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in MEASUREMENT_COLUMNS:

        current_values = (
            df[col]
            .to_numpy()
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY validated zero restorations."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# CRITICAL DIETARY FIDELITY RULES:
#
# NO:
#
# round()
# float_format
# format strings
# Decimal quantization
# recalculation
#
# Let pandas write the original decoded float.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 24. RE-READ WITH ROUND-TRIP FLOAT PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 26. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 164 columns numerically match "
    "the expected XPT values exactly."
)


# ============================================================
# 27. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 28. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(
        reader
    )


if len(raw_rows) != 9762:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 29. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# Integer fields:
#
# expected:
# str(int(source_value))
#
#
# Dietary measurement fields:
#
# expected:
# str(float(source_value))
#
# This preserves pandas' actual decoded
# source representation.
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        if col in MEASUREMENT_COLUMNS:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    float(source_value)
                )
            )

        else:

            expected_text = (
                ""
                if pd.isna(source_value)
                else str(
                    int(source_value)
                )
            )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:

        print(
            example
        )


    raise ValueError(
        "DR1TOT_E raw CSV fidelity failure."
    )


print(
    "PASS: Raw CSV tokens exactly match "
    "expected source representation."
)


# ============================================================
# 30. UNWANTED .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# IMPORTANT:
#
# Do NOT globally reject ".0".
#
# Dietary measurement fields are allowed to have .0.
#

unwanted_dot_zero_count = 0
unwanted_dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = (
            row[col]
        )


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            unwanted_dot_zero_count += 1


            if len(
                unwanted_dot_zero_examples
            ) < 20:

                unwanted_dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nUnwanted integer '.0' values:",
    unwanted_dot_zero_count
)


if unwanted_dot_zero_count != 0:

    print(
        unwanted_dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 remains in "
        "SEQN/code/count fields."
    )


print(
    "PASS: No unwanted .0 in integer fields."
)


# ============================================================
# 31. STRICT MEASUREMENT RAW CHECK
# ============================================================

#
# This explicitly verifies EVERY dietary
# measurement cell against:
#
# str(float(XPT source value))
#

measurement_raw_mismatches = 0
measurement_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in MEASUREMENT_COLUMNS:

        value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(value)
            else str(
                float(value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            measurement_raw_mismatches += 1


            if len(
                measurement_examples
            ) < 20:

                measurement_examples.append(
                    {
                        "row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "Dietary measurement raw differences:",
    measurement_raw_mismatches
)


if measurement_raw_mismatches != 0:

    print(
        measurement_examples
    )

    raise ValueError(
        "Dietary decimal representation changed."
    )


print(
    "PASS: Every dietary measurement value "
    "is preserved AS IS."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = (
        f.read()
    )


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings in CSV:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 34. FINAL ZERO COUNTS
# ============================================================

zero_failures = {}


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        csv_df[col]
        .eq(0)
        .sum()
    )


    if actual_count != expected_count:

        zero_failures[col] = {
            "expected": expected_count,
            "actual": actual_count
        }


if zero_failures:

    raise ValueError(
        "Final zero counts changed:\n"
        + str(
            zero_failures
        )
    )


print(
    "PASS: All 67,100 restored zero values "
    "remain correct after CSV export."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DR1TOT_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)

print(
    "\nOriginal tiny artifacts:",
    f"{EXPECTED_TINY_TOTAL:,}"
)

print(
    "Variables affected:",
    len(
        EXPECTED_TINY_COUNTS
    )
)

print(
    "Remaining tiny artifacts:",
    remaining_tiny
)


print(
    "\nSpecial zero fields:"
)

print(
    "DR1DBIH = 432 zeros"
)

print(
    "DR1TNUMF = 1 zero"
)


print(
    "\nDietary measurement fields preserved AS IS:",
    len(
        MEASUREMENT_COLUMNS
    )
)

print(
    "Nutrient fields preserved AS IS:",
    len(
        NUTRIENT_COLUMNS
    )
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Dietary measurement raw differences:",
    measurement_raw_mismatches
)

print(
    "Unwanted integer .0 values:",
    unwanted_dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "DR1TOT_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 DR1TOT_E XPT -> CSV
Rows: 9,762
Columns: 164
PASS: DR1TOT_E structure = 9,762 rows x 164 columns.

Dietary measurement fields preserved AS IS: 70
Nutrient fields: 65
Integer/code/count fields: 94

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 67,100

Tiny values by variable:
DR1TATOA    8184
DR1TALCO    7666
DR1BWATZ    6034
DR1TB12A    5916
DR1TP184    5776
DR1_330Z    5264
DR1TTHEO    4472
DR1TP225    3576
DR1TM221    2879
DR1_320Z    2662
DR1TP205    2461
DR1TLYCO    2433
DR1TCAFF    2179
DR1TP226    2005
DR1TACAR    1161
DR1TCRYP     994
DR1TS040     554
DR1TS060     503
DR1DBIH      432
DR1TP204     346
DR1TS080     293
DR1TFA       144
DR1TBCAR     128
DR1TLZ       118
DR1TVD       112
DR1TM201     102
DR1TS100     100
DR1TFIBE      98
DR1TFF        82
DR1TRET       79
DR1TCHOL      53
DR1TVC        42
DR1TM161      34
DR1TS120      30
DR1TVARA      22
DR1TVB12      22
DR1TS140      12
DR1TVK        10
DR1TATOC       9
DR1TM181       9
DR1